# Optimisation courte autour de b4 — Maick Dane Nkou

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maick-code/airf-multilingual-tokenizer-challenge/blob/arena/01a09f96-airf-multilingual-tokenizer-ch/notebooks/optimisation_b4.ipynb)

**Notebook expérimental séparé — aucune modification de la soumission.**

Votre dernier résultat Colab sur les 24 000 lignes officielles : **b4 = 1,939667**, zéro UNK,
reconstruction 100 %, zéro pénalité, marge EN/FR ≈ **1,79 %**.
SHA-256 de ce fichier : `b2f9a461ce1b0e8bff07c00d982614f49d555464b8fb1efa46e66b08b67013e4`.
Ces valeurs sont des références historiques : **aucun résultat d'optimisation n'est prérempli**.
Le tokenizer actuellement commité est toujours `weights-yo-am4`, précédemment mesuré à 1,963189.

## Plan validé : score + prudence, six configurations, Colab gratuit

| Candidat | en | fr | ha | sw | yo | am |
|---|---:|---:|---:|---:|---:|---:|
| b4-reference | 1 | 1 | 4 | 4 | 4 | 4 |
| am5 | 1 | 1 | 4 | 4 | 4 | 5 |
| yo5 | 1 | 1 | 4 | 4 | 5 | 4 |
| yo5-am5 | 1 | 1 | 4 | 4 | 5 | 5 |
| ha3-am5 | 1 | 1 | 3 | 4 | 4 | 5 |
| fr125-am5 | 1 | 1,25 | 4 | 4 | 4 | 5 |

Ce sont des **hypothèses**, pas des améliorations garanties. L'amharique coûte davantage de tokens
par mot, mais lui attribuer davantage de vocabulaire peut dégrader les autres langues.
Le dernier essai teste un léger soutien au français : une copie supplémentaire de chaque quatrième
ligne française de **train**, sans choisir les textes par leur contenu. Tous les originaux restent présents.
La grille est fixée avant la recherche : aucun essai adaptatif supplémentaire.

Architecture inchangée : BPE 10 000, min_frequency 5, `space_word` + ByteLevel réversible,
alphabet byte complet, aucun normaliseur, token spécial, corpus externe ou modèle préentraîné.
Entraînement sur les **240 000 lignes train uniquement** ; validation uniquement pour évaluer et sélectionner.

### Critère de recommandation (distinct du score officiel)
Une alternative doit avoir un score complet **strictement inférieur au b4 réévalué**, zéro UNK,
reconstruction exacte, zéro pénalité et une marge EN/FR **au moins égale à celle de ce b4**.
Le seuil utilise sa valeur exacte, pas 1,8 % arrondi et pas l'ancienne règle de 5 %.
Tous les scores et pénalités restent visibles, y compris pour les alternatives non recommandées.
Si aucun essai ne remplit ces conditions, on conserve b4 comme référence de cette recherche.
Cela ne remplace **pas** le tokenizer soumis.

### Avant « Tout exécuter »
1. Choisir un environnement **CPU** ; le GPU n'accélère pas ce BPE.
2. Pour conserver les modèles après la destruction de la session, mettre **`USE_GOOGLE_DRIVE = True`**
   dans la configuration et autoriser le montage. Sinon, les sauvegardes Colab sont temporaires.
3. Exécuter toutes les cellules. Un seul entraînement à la fois, dans un processus séparé ;
   les répétitions sont générées à la volée, pas stockées en mémoire.
4. Après interruption, rouvrir ce notebook avec les mêmes options et relancer : les modèles terminés
   et vérifiés sont réutilisés puis **réévalués**. Un entraînement interrompu repart de zéro ; ce n'est pas
   une reprise à l'intérieur de l'algorithme BPE.
5. Partager `comparison.csv` ou `review_reports.zip`. Aucun téléchargement, export de soumission,
   commit ou remplacement de modèle n'est automatique.

La durée totale n'est pas connue à l'avance : elle dépend surtout des six entraînements et de la RAM
allouée par Colab. Le premier essai mesure ce coût. En cas d'erreur mémoire, le journal est conservé ;
ne pas masquer l'erreur en évaluant un corpus réduit.

In [ ]:
# Dépendances minimales. Aucun package d'optimisation GPU n'est nécessaire.
import importlib.util
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

TOKENIZERS_VERSION = "0.22.1"
def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return ""

requirements = []
if installed("tokenizers") != TOKENIZERS_VERSION:
    requirements.append(f"tokenizers=={TOKENIZERS_VERSION}")
if installed("datasets").split(".")[0] != "4":
    requirements.append("datasets>=4.0,<5")
if requirements:
    if importlib.util.find_spec("pip"):
        command = [sys.executable, "-m", "pip", "install", "-q", *requirements]
    elif shutil.which("uv"):
        command = ["uv", "pip", "install", "--python", sys.executable, *requirements]
    else:
        raise RuntimeError("Installer pip ou uv avant de poursuivre")
    subprocess.run(command, check=True)
import tokenizers
if tokenizers.__version__ != TOKENIZERS_VERSION:
    raise RuntimeError("Redémarrer la session après la mise à jour de tokenizers")
print("tokenizers :", tokenizers.__version__, "| datasets :", version("datasets"))

In [ ]:
# Options à choisir AVANT Tout exécuter.
USE_GOOGLE_DRIVE = False  # True conseillé sur Colab pour survivre à la perte de session.
RUN_LABEL = "balanced-v1"  # Garder identique pour reprendre. Changer pour une nouvelle recherche indépendante.
CPU_THREADS = 2

import csv
import gc
import hashlib
import html
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

TOKENIZERS_VERSION = "0.22.1"
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
DATASET_ID = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"
CHECKER_COMMIT = "75578f2400c39b1f8e31ce7e7104b37fbc470d11"
CHECKER_SHA256 = "1727de34136097eb48addabf90501589bdfefa31c20e201bab53c38f2f7c9688"
CHECKER_URL = f"https://raw.githubusercontent.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge/{CHECKER_COMMIT}/starter/utils.py"
HISTORICAL_B4_SHA256 = "b2f9a461ce1b0e8bff07c00d982614f49d555464b8fb1efa46e66b08b67013e4"

# Poids exprimés en quarts pour éviter une arithmétique flottante dans les répétitions.
# Ordre : en, fr, ha, sw, yo, am. 4 unités = x1 ; 5 unités = x1,25.
WEIGHT_GRID = [
    ("b4-reference", (4, 4, 16, 16, 16, 16)),
    ("am5",          (4, 4, 16, 16, 16, 20)),
    ("yo5",          (4, 4, 16, 16, 20, 16)),
    ("yo5-am5",      (4, 4, 16, 16, 20, 20)),
    ("ha3-am5",      (4, 4, 12, 16, 16, 20)),
    ("fr125-am5",    (4, 5, 16, 16, 16, 20)),
]
CONFIGS = [{"name": name, "quarter_units": dict(zip(LANGUAGES, units, strict=True)),
            "vocab_size": 10_000, "min_frequency": 5, "architecture": "space_word_v1"}
           for name, units in WEIGHT_GRID]
if not re.fullmatch(r"[a-zA-Z0-9_-]+", RUN_LABEL):
    raise ValueError("RUN_LABEL doit être un nom simple, sans chemin")
if type(CPU_THREADS) is not int or not 1 <= CPU_THREADS <= 4:
    raise ValueError("Choisir 1 à 4 threads CPU")
os.environ["RAYON_NUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"
for config in CONFIGS:
    repeats = {l: config["quarter_units"][l] / 4 for l in LANGUAGES}
    weighted_rows = 40_000 * sum(config["quarter_units"].values()) // 4
    print(config["name"], repeats, f"— {weighted_rows:,} lignes pondérées")

In [ ]:
# Les corpus/cache restent locaux ; Drive ne reçoit que les petits modèles et rapports.
roots = (Path.cwd(), *Path.cwd().parents)
repo_root = next((p for p in roots if (p / "pyproject.toml").is_file() and (p / "submissions").is_dir()), None)
workspace = repo_root or Path.cwd()
WORK_DIR = workspace / "artifacts" / "optimisation-b4-runtime"
WORK_DIR.mkdir(parents=True, exist_ok=True)
SAVE_ROOT = workspace / "artifacts" / "optimisation-b4"
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Google Drive nécessite Colab ; sinon laisser USE_GOOGLE_DRIVE=False") from exc
    drive.mount("/content/drive")
    SAVE_ROOT = Path("/content/drive/MyDrive/airf-tokenizer/optimisation-b4")
else:
    print("ATTENTION : sans Drive, sauvegardes locales seulement. Les télécharger avant de fermer Colab.")
for directory in (WORK_DIR, SAVE_ROOT):
    if "submissions" in directory.resolve().parts:
        raise ValueError("Les sorties ne doivent jamais être placées dans submissions/")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = WORK_DIR / "train.jsonl"
VALIDATION_PATH = WORK_DIR / "validation.jsonl"
WORKER_PATH = WORK_DIR / "cpu_worker.py"
CHECKER_PATH = WORK_DIR / "official_utils.py"
print("Calcul/cache :", WORK_DIR)
print("Sauvegardes :", SAVE_ROOT)

## Travailleur CPU isolé et contrôles de reconstruction

Le code ci-dessous est intégralement dans le notebook ; il est écrit dans un fichier temporaire local
pour libérer la mémoire native à la fin de chaque processus. **Le processus d'entraînement ne reçoit
aucun chemin vers la validation.** Le processus d'évaluation charge le checker officiel, vérifie son
empreinte, appelle directement `profile_submission`, puis contrôle aussi les espaces et Unicode exactement.
La seule réduction de coût du checker est `repeats=1` pour son benchmark de vitesse, informatif :
les **24 000 lignes** et le calcul du score ne sont pas réduits.

In [ ]:
WORKER_SOURCE = r'''"""Isolated CPU worker: train-only BPE, or pinned official evaluation."""
import argparse
import hashlib
import importlib.util
import json
import os
import time
from collections import Counter
from pathlib import Path

from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers, trainers

LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
TOKENIZERS_VERSION = "0.22.1"
CHECKER_SHA256 = "1727de34136097eb48addabf90501589bdfefa31c20e201bab53c38f2f7c9688"


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def read_rows(path):
    with Path(path).open(encoding="utf-8") as stream:
        for line in stream:
            language, text = json.loads(line)
            if language not in LANGUAGES or not isinstance(text, str) or not text.split():
                raise ValueError("Invalid corpus row")
            yield language, text


def weighted_texts(rows, units):
    """Quarter units: 4 = x1; 5 = x1.25. Every original row appears at least once.

    Extra copies are distributed by within-language row index, never by content
    or validation statistics. With 40,000 rows, all configured ratios are exact.
    """
    if set(units) != set(LANGUAGES) or any(type(n) is not int or n < 4 for n in units.values()):
        raise ValueError("Expected six integer weights >= 4 quarter units")
    seen = Counter()
    for language, text in rows:
        index = seen[language]
        seen[language] += 1
        copies = ((index + 1) * units[language]) // 4 - (index * units[language]) // 4
        for _ in range(copies):
            yield text


def train_tokenizer(texts, *, vocab_size=10_000, min_frequency=5, length=None):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Split(Regex(r" ?\S+|\s+"), behavior="isolated"),
        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
    ])
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=min_frequency, special_tokens=[],
        initial_alphabet=sorted(pre_tokenizers.ByteLevel.alphabet()), show_progress=True,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer, length=length)
    return tokenizer


ROUNDTRIP_CASES = [
    "", " ", "   ", "\t\n\r\n", "  Hello  WORLD!\tNext\nline.  ",
    "[UNK] [CLS] [SEP] <0xFF>", "é e\u0301 Ì I\u0300", "👩🏿‍💻 🌍 中文 العربية",
    "a\u00a0b\u2003c\u200bd", "\x00\x01\x7f\ufeff\U0010ffff", "don't l’amour — … ።",
    "Knowledge grows when it is shared.", "Le savoir grandit lorsqu’il est partagé.",
    "Ilimi yana ƙaruwa idan an raba shi.", "Maarifa hukua yanaposhirikishwa.",
    "Ìmọ̀ ń pọ̀ sí i nígbà tí a bá pín in.", "እውቀት ሲካፈል ያድጋል።",
]


def roundtrip_failures(tokenizer, texts, batch_size=256):
    failed = 0
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        ids = [enc.ids for enc in tokenizer.encode_batch(batch, add_special_tokens=False)]
        decoded = tokenizer.decode_batch(ids, skip_special_tokens=False)
        skipped = tokenizer.decode_batch(ids, skip_special_tokens=True)
        failed += sum(a != b or a != c for a, b, c in zip(batch, decoded, skipped, strict=True))
    return failed


def train_job(train_path, config_path, output):
    config = json.loads(Path(config_path).read_text(encoding="utf-8"))
    if config["vocab_size"] != 10_000 or config["min_frequency"] != 5 or config["architecture"] != "space_word_v1":
        raise ValueError("Unexpected search recipe")
    counts = Counter(language for language, _ in read_rows(train_path))
    if counts != dict.fromkeys(LANGUAGES, 40_000):
        raise ValueError("Full official train required: 40,000 rows per language")
    before = sha256_file(train_path)
    start = time.perf_counter()
    tokenizer = train_tokenizer(
        weighted_texts(read_rows(train_path), config["quarter_units"]),
        length=sum(counts[l] * config["quarter_units"][l] // 4 for l in LANGUAGES),
    )
    if tokenizer.get_vocab_size(with_added_tokens=True) != 10_000:
        raise ValueError("Expected exactly 10,000 vocabulary entries")
    output = Path(output)
    temporary = output.with_name(output.name + ".partial")
    tokenizer.save(str(temporary), pretty=True)
    del tokenizer
    reloaded = Tokenizer.from_file(str(temporary))
    if roundtrip_failures(reloaded, ROUNDTRIP_CASES):
        raise ValueError("Lossy pipeline; candidate not saved")
    if before != sha256_file(train_path):
        raise ValueError("Training corpus changed during training")
    os.replace(temporary, output)
    atomic_json(output.parent / "training.json", {
        "config": config, "train_sha256": before, "tokenizer_sha256": sha256_file(output),
        "training_seconds": time.perf_counter() - start,
        "weighted_rows": sum(counts[l] * config["quarter_units"][l] // 4 for l in LANGUAGES),
    })


def load_checker(path):
    if sha256_file(path) != CHECKER_SHA256:
        raise ValueError("Official checker SHA-256 mismatch")
    spec = importlib.util.spec_from_file_location("pinned_official_checker", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    if module.REQUIRED_TOKENIZERS_VERSION != TOKENIZERS_VERSION or module.RECONSTRUCTION_PENALTY != 3.0:
        raise ValueError("Unexpected checker contract")
    return module


def evaluate_job(model_path, validation_path, checker_path, output_dir):
    output_dir = Path(output_dir)
    rows = list(read_rows(validation_path))
    if Counter(l for l, _ in rows) != dict.fromkeys(LANGUAGES, 4_000):
        raise ValueError("Full official validation required: 4,000 rows per language")
    before = sha256_file(model_path)
    helper = load_checker(checker_path)
    # Unmodified official score, including BOTH penalties. No custom 5% gate.
    report = helper.profile_submission(model_path, data=rows, repeats=1, verbose=True)
    # Preserve official output BEFORE any additional reconstruction/selection checks.
    atomic_json(output_dir / "official_report.json", report)
    if sha256_file(model_path) != before:
        raise ValueError("Tokenizer changed during evaluation")
    strict = None
    if report.get("valid"):
        tokenizer = Tokenizer.from_file(str(model_path))
        strict = roundtrip_failures(tokenizer, [text for _, text in rows] + ROUNDTRIP_CASES)
    atomic_json(output_dir / "evaluation.json", {
        "official": report, "strict_failures": strict,
        "tokenizer_sha256": before, "validation_sha256": sha256_file(validation_path),
        "checker_sha256": CHECKER_SHA256,
    })


def main():
    from tokenizers import __version__
    if __version__ != TOKENIZERS_VERSION:
        raise RuntimeError("tokenizers==0.22.1 is required")
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="command", required=True)
    train = sub.add_parser("train")
    for name in ("train_path", "config_path", "output"):
        train.add_argument(name)
    evaluate = sub.add_parser("evaluate")
    for name in ("model_path", "validation_path", "checker_path", "output_dir"):
        evaluate.add_argument(name)
    args = vars(parser.parse_args())
    command = args.pop("command")
    (train_job if command == "train" else evaluate_job)(**args)


if __name__ == "__main__":
    main()
'''

In [ ]:
# Installation locale du travailleur, sans téléchargement de modèle.
WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
spec = importlib.util.spec_from_file_location("optimisation_b4_worker", WORKER_PATH)
worker = importlib.util.module_from_spec(spec)
spec.loader.exec_module(worker)
sha256_file = worker.sha256_file
atomic_json = worker.atomic_json

# Réutiliser seulement un checker dont le SHA est exactement celui attendu.
helper = None
for path in (CHECKER_PATH, repo_root / "starter/utils.py" if repo_root else None):
    if path is not None and path.is_file() and sha256_file(path) == CHECKER_SHA256:
        helper = path.read_bytes()
        break
if helper is None:
    with urllib.request.urlopen(CHECKER_URL, timeout=60) as response:
        helper = response.read()
if hashlib.sha256(helper).hexdigest() != CHECKER_SHA256:
    raise RuntimeError("SHA du checker incorrect ; arrêt sans entraînement")
CHECKER_PATH.write_bytes(helper)
worker.load_checker(CHECKER_PATH)
print("Checker officiel vérifié :", CHECKER_COMMIT, "| SHA-256 :", CHECKER_SHA256)

In [ ]:
# Test jetable : ses exemples et ses merges ne servent JAMAIS au modèle de compétition.
from tokenizers import Tokenizer, decoders, normalizers, pre_tokenizers
mini = worker.train_tokenizer(worker.ROUNDTRIP_CASES, vocab_size=512, min_frequency=1)
with tempfile.TemporaryDirectory(dir=WORK_DIR) as tmp:
    path = Path(tmp) / "tokenizer.json"
    mini.save(str(path))
    mini = Tokenizer.from_file(str(path))
    assert worker.roundtrip_failures(mini, worker.ROUNDTRIP_CASES) == 0
    for mutation in ("lowercase", "whitespace_split", "wrong_decoder"):
        broken = Tokenizer.from_str(mini.to_str())
        if mutation == "lowercase":
            broken.normalizer = normalizers.Lowercase()
        elif mutation == "whitespace_split":
            broken.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
        else:
            broken.decoder = decoders.ByteFallback()
        assert worker.roundtrip_failures(broken, worker.ROUNDTRIP_CASES) > 0, mutation
        del broken
del mini
gc.collect()
print("Tests de reconstruction réussis, y compris rejet de trois variantes avec perte.")

## Données publiques uniquement — corpus complets

Version fixée à `v1.0.0`. Si le téléchargement échoue, la cellule échoue : **aucun remplacement par un
jeu de démonstration**. Les textes originaux ne sont ni nettoyés ni normalisés. Le corpus d'entraînement
est sérialisé dans l'ordre round-robin en/fr/ha/sw/yo/am, en gardant l'ordre interne de chaque langue,
comme dans la recette b4. Les répétitions sont calculées plus tard par le travailleur.

Les comptes, empreintes Hugging Face et SHA-256 des corpus sérialisés sont conservés pour la reprise.
Le SHA d'un JSONL dépend de ce format de sérialisation : ne pas le comparer à un hash d'audit utilisant
un autre format. Les corpus et les caches ne sont **pas** copiés dans Google Drive ni dans Git.

In [ ]:
def dump_row(stream, language, text):
    stream.write(json.dumps([language, text], ensure_ascii=False, separators=(",", ":")) + "\n")


def prepare_split(split, destination):
    from datasets import load_dataset
    if split not in {"train", "validation"}:
        raise ValueError("Seulement train et validation")
    dataset = load_dataset(DATASET_ID, revision=DATASET_REVISION, split=split,
                           cache_dir=str(WORK_DIR / "dataset-cache"))
    per_language = 40_000 if split == "train" else 4_000
    fingerprint = dataset._fingerprint
    destination = Path(destination)
    temporary = destination.with_name(destination.name + ".partial")
    counts = Counter()
    with tempfile.TemporaryDirectory(dir=WORK_DIR) as tmp:
        paths = {l: Path(tmp) / f"{l}.jsonl" for l in LANGUAGES}
        streams = {l: p.open("w", encoding="utf-8", newline="\n") for l, p in paths.items()} if split == "train" else {}
        try:
            with temporary.open("w", encoding="utf-8", newline="\n") as output:
                for row in dataset:
                    language, text = row["language"], row["text"]
                    if language not in LANGUAGES or not isinstance(text, str) or not text.split():
                        raise ValueError(f"Ligne invalide dans {split}")
                    counts[language] += 1
                    dump_row(streams[language] if split == "train" else output, language, text)
                if counts != dict.fromkeys(LANGUAGES, per_language):
                    raise ValueError(f"Comptes inattendus pour {split}: {dict(counts)}")
                if split == "train":
                    for stream in streams.values():
                        stream.close()
                    readers = [paths[l].open(encoding="utf-8") for l in LANGUAGES]
                    try:
                        for group in zip(*readers, strict=True):
                            output.writelines(group)
                    finally:
                        for reader in readers:
                            reader.close()
        finally:
            for stream in streams.values():
                stream.close()
    os.replace(temporary, destination)
    result = {"split": split, "rows": sum(counts.values()), "counts": dict(counts),
              "fingerprint": fingerprint, "sha256": sha256_file(destination),
              "serialization": "jsonl-language-text-v1; train round-robin; validation source order"}
    del dataset
    gc.collect()
    print(split, result)
    return result

In [ ]:
train_info = prepare_split("train", TRAIN_PATH)
validation_info = prepare_split("validation", VALIDATION_PATH)
CONTEXT = {
    "policy_version": "balanced-b4-v1", "dataset": DATASET_ID, "revision": DATASET_REVISION,
    "train": train_info, "validation": validation_info,
    "tokenizers_version": tokenizers.__version__, "datasets_version": version("datasets"),
    "python_version": sys.version.split()[0], "cpu_threads": CPU_THREADS,
    "checker_commit": CHECKER_COMMIT, "checker_sha256": CHECKER_SHA256,
    "worker_sha256": sha256_file(WORKER_PATH), "configs": CONFIGS,
}
context_id = hashlib.sha256(json.dumps(CONTEXT, sort_keys=True).encode("utf-8")).hexdigest()[:16]
SESSION_DIR = SAVE_ROOT / f"{RUN_LABEL}-{context_id}"
SESSION_DIR.mkdir(parents=True, exist_ok=True)
atomic_json(SESSION_DIR / "run_context.json", CONTEXT)
print("Dossier de cette recherche :", SESSION_DIR)
print("Si les données, le code de travailleur ou la configuration changent, un nouveau dossier est utilisé.")

## Recherche bornée et reprise vérifiée

Six entraînements complets au maximum dans une exécution neuve (dont la référence).
Chaque candidat terminé est sauvegardé avec sa configuration, son temps d'entraînement, ses SHA et
un marqueur de complétion écrit en dernier. Lors d'une reprise, le fichier doit correspondre à ce reçu ;
une corruption entraîne une erreur explicite, jamais un remplacement silencieux.

Le score officiel est enregistré **avant** nos critères de sélection. Les pénalités ne sont jamais supprimées.
Une erreur sur une alternative est enregistrée et n'empêche pas les essais suivants ; une référence invalide
arrête la recherche. Si le nouveau b4 diffère des anciens octets ou du score historique, le notebook l'affiche
et utilise **uniquement ses nouvelles mesures** pour comparer les alternatives.

La validation a déjà servi à plusieurs choix de recettes. Les gains mesurés ici peuvent donc être optimistes ;
ce n'est pas une estimation indépendante du test caché. Aucun gain minimal ou intervalle de confiance n'est
inventé, et aucun seuil de marge ne garantit le comportement sur des textes différents.

In [ ]:
# Search policy and resumable execution — no dataset downloads or training in this cell.
def metric_summary(evaluation):
    """Validate score arithmetic; never subtract or hide an official penalty."""
    report = evaluation["official"]
    if not report.get("valid") or report.get("rows") != 24_000:
        raise ValueError("Invalid file or incomplete official evaluation")
    if report.get("vocab_size") != 10_000 or report.get("tokenizers_version") != TOKENIZERS_VERSION:
        raise ValueError("Unexpected vocabulary or runtime")
    for key in ("fertility", "unknown_rate", "penalised"):
        if set(report.get(key, {})) != set(LANGUAGES):
            raise ValueError(f"Missing language in {key}")
        if any(not math.isfinite(v) or v < 0 for v in report[key].values()):
            raise ValueError(f"Invalid {key}")
    for key in ("score", "guardrail_budget", "guardrail_penalty", "reconstruction_penalty", "reconstruction", "lossy_rows"):
        if not math.isfinite(report[key]) or report[key] < 0:
            raise ValueError(f"Invalid {key}")
    for lang in LANGUAGES:
        expected = report["fertility"][lang] + 100 * report["unknown_rate"][lang]
        if not math.isclose(report["penalised"][lang], expected, rel_tol=1e-12, abs_tol=1e-12):
            raise ValueError("Unknown penalty arithmetic mismatch")
    base = sum(report["penalised"][l] for l in SCORED_LANGUAGES) / 4
    budget = 1.15 * sum(report["fertility"][l] for l in SCORED_LANGUAGES) / 4
    penalty = sum(max(0, report["fertility"][l] - budget) for l in ("en", "fr"))
    reconstruction = 1 - report["lossy_rows"] / 24_000
    if budget <= 0 or not 0 <= reconstruction <= 1:
        raise ValueError("Invalid reconstruction or guardrail budget")
    expected_values = {
        "guardrail_budget": budget, "guardrail_penalty": penalty,
        "reconstruction": reconstruction, "reconstruction_penalty": 3 * (1 - reconstruction),
        "score": base + penalty + 3 * (1 - reconstruction),
    }
    for key, expected in expected_values.items():
        if not math.isclose(report[key], expected, rel_tol=1e-12, abs_tol=1e-12):
            raise ValueError(f"Official arithmetic mismatch: {key}")
    return {
        "base": base, "score": report["score"], "guardrail_penalty": penalty,
        "reconstruction_penalty": report["reconstruction_penalty"],
        "headroom": min((budget - report["fertility"][l]) / budget for l in ("en", "fr")),
        "lossless": report["lossy_rows"] == 0 and evaluation.get("strict_failures") == 0,
        "zero_unk": all(v == 0 for v in report["unknown_rate"].values()),
    }


def selection_reasons(evaluation, baseline):
    measured, reference = metric_summary(evaluation), metric_summary(baseline)
    reasons = []
    if not measured["lossless"]:
        reasons.append("reconstruction non exacte")
    if not measured["zero_unk"]:
        reasons.append("tokens inconnus")
    if measured["guardrail_penalty"] != 0 or measured["reconstruction_penalty"] != 0:
        reasons.append("pénalité officielle")
    # The measured, unrounded reference margin, NOT a hard-coded 1.8% or 5%.
    if measured["headroom"] < reference["headroom"] - 1e-12:
        reasons.append("marge inférieure au b4 réévalué")
    if measured["score"] >= reference["score"] - 1e-12:
        reasons.append("pas de baisse du score")
    return reasons


def verify_saved_candidate(directory, expected):
    """Missing completion marker = interrupted job. Corruption = explicit error."""
    directory = Path(directory)
    manifest = directory / "manifest.json"
    if not manifest.exists():
        return None
    if json.loads(manifest.read_text(encoding="utf-8")) != expected:
        raise ValueError("Saved provenance mismatch; use a different RUN_LABEL, do not overwrite")
    receipt = json.loads((directory / "training.json").read_text(encoding="utf-8"))
    if receipt["config"] != expected["config"] or receipt["train_sha256"] != expected["context"]["train"]["sha256"]:
        raise ValueError("Training receipt mismatch")
    if sha256_file(directory / "tokenizer.json") != receipt["tokenizer_sha256"]:
        raise ValueError("Saved tokenizer SHA-256 mismatch; refusing reuse")
    return receipt


def run_worker(arguments, log_path):
    env = os.environ.copy()
    env.update(RAYON_NUM_THREADS=str(CPU_THREADS), TOKENIZERS_PARALLELISM="true", PYTHONUNBUFFERED="1")
    command = [sys.executable, "-u", str(WORKER_PATH), *map(str, arguments)]
    with Path(log_path).open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", env=env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            if process.wait() != 0:
                raise RuntimeError(f"Worker failed ({process.returncode}); see {log_path}")
        finally:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            process.stdout.close()


def atomic_copy(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".partial")
    shutil.copyfile(source, temporary)
    if sha256_file(source) != sha256_file(temporary):
        raise ValueError("Copy checksum mismatch")
    os.replace(temporary, destination)


def run_candidate(config, context, session_dir):
    directory = Path(session_dir) / config["name"]
    directory.mkdir(parents=True, exist_ok=True)
    expected = {"context": context, "config": config}
    receipt = verify_saved_candidate(directory, expected)
    stage = WORK_DIR / "staging" / config["name"]
    stage.mkdir(parents=True, exist_ok=True)
    model_path = stage / "tokenizer.json"
    if receipt is None:
        atomic_json(stage / "config.json", config)
        print(f"\n=== Entraînement : {config['name']} ===", flush=True)
        try:
            run_worker(["train", TRAIN_PATH, stage / "config.json", model_path], stage / "training.log")
        finally:
            if (stage / "training.log").exists():
                atomic_copy(stage / "training.log", directory / "training.log")
        receipt = json.loads((stage / "training.json").read_text(encoding="utf-8"))
        if (receipt["config"] != config or receipt["train_sha256"] != context["train"]["sha256"]
                or receipt["tokenizer_sha256"] != sha256_file(model_path)):
            raise ValueError("Worker training provenance mismatch")
        atomic_copy(model_path, directory / "tokenizer.json")
        atomic_json(directory / "training.json", receipt)
        # Completion marker is written LAST, only after both saved files exist.
        atomic_json(directory / "manifest.json", expected)
    else:
        print(f"\n=== Reprise sans entraînement : {config['name']} ===", flush=True)
        atomic_copy(directory / "tokenizer.json", model_path)
    if model_path.stat().st_size > 20 * 1024 * 1024:
        raise ValueError("Tokenizer exceeds 20 MiB")
    # Always re-evaluate reused files: saved scores are not trusted as current results.
    # Clear only local scratch evaluation files to prevent stale reports after a crash.
    for name in ("official_report.json", "evaluation.json"):
        (stage / name).unlink(missing_ok=True)
    try:
        run_worker(["evaluate", model_path, VALIDATION_PATH, CHECKER_PATH, stage], stage / "evaluation.log")
    finally:
        for name in ("official_report.json", "evaluation.log"):
            if (stage / name).exists():
                atomic_copy(stage / name, directory / name)
    evaluation = json.loads((stage / "evaluation.json").read_text(encoding="utf-8"))
    if (evaluation["tokenizer_sha256"] != receipt["tokenizer_sha256"]
            or evaluation["validation_sha256"] != context["validation"]["sha256"]
            or evaluation["checker_sha256"] != CHECKER_SHA256
            or sha256_file(directory / "tokenizer.json") != receipt["tokenizer_sha256"]):
        raise ValueError("Evaluation provenance mismatch")
    result = {**evaluation, "config": config, "training_seconds": receipt["training_seconds"],
              "weighted_rows": receipt["weighted_rows"], "directory": str(directory)}
    atomic_json(directory / "evaluation.json", result)
    return result


def choose_best(history):
    if not history or history[0].get("status") != "reference":
        raise ValueError("A verified baseline must be first")
    best = history[0]
    for entry in history[1:]:
        if entry.get("status") == "measured" and not selection_reasons(entry, history[0]):
            if metric_summary(entry)["score"] < metric_summary(best)["score"] - 1e-12:
                best = entry
    return best


def save_progress(history, context, session_dir):
    best = choose_best(history) if history and history[0].get("status") == "reference" else None
    atomic_json(Path(session_dir) / "search_history.json", {
        "context": context, "candidates": history,
        "best": best["config"]["name"] if best else None,
        "minimum_headroom": metric_summary(history[0])["headroom"] if best else None,
        "selection_rule": "lower full score; zero penalties/UNK; exact reconstruction; no less headroom than remeasured b4",
        "review_only": True,
    })
    columns = ["candidate", "status", "score", "delta_vs_b4", "headroom_percent", "base",
               "guardrail_penalty", "reconstruction_penalty", "lossless", "zero_unk",
               *[f"fertility_{l}" for l in LANGUAGES], "recommended", "reason", "sha256"]
    destination = Path(session_dir) / "comparison.csv"
    with destination.with_suffix(".csv.partial").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        for entry in history:
            row = {"candidate": entry["config"]["name"], "status": entry["status"],
                   "reason": entry.get("error", ""), "recommended": entry is best}
            if entry["status"] in {"reference", "measured"}:
                m = metric_summary(entry)
                row.update({k: m[k] for k in ("score", "base", "guardrail_penalty", "reconstruction_penalty", "lossless", "zero_unk")})
                row.update(delta_vs_b4=m["score"] - metric_summary(history[0])["score"],
                           headroom_percent=100 * m["headroom"], sha256=entry["tokenizer_sha256"])
                row.update({f"fertility_{l}": entry["official"]["fertility"][l] for l in LANGUAGES})
                row["reason"] = "référence" if entry["status"] == "reference" else "; ".join(selection_reasons(entry, history[0])) or "amélioration admissible"
            writer.writerow(row)
    os.replace(destination.with_suffix(".csv.partial"), destination)


def run_search(configs, context, session_dir):
    if len(configs) != 6 or configs[0]["name"] != "b4-reference":
        raise ValueError("Exactly six predeclared configurations, baseline first")
    if configs[0]["quarter_units"] != dict(zip(LANGUAGES, (4, 4, 16, 16, 16, 16))):
        raise ValueError("Baseline must be the original b4 recipe")
    for path, split in ((TRAIN_PATH, "train"), (VALIDATION_PATH, "validation")):
        if sha256_file(path) != context[split]["sha256"]:
            raise ValueError(f"{split} changed after provenance capture")
    history = []
    for config in configs:
        try:
            entry = run_candidate(config, context, session_dir)
            measured = metric_summary(entry)
            if not history:
                if not measured["lossless"] or not measured["zero_unk"] or measured["guardrail_penalty"] != 0:
                    raise ValueError("Baseline not lossless/zero-UNK/penalty-free; review before search")
                entry["status"] = "reference"
                print(f"\nB4 RÉÉVALUÉ : {measured['score']:.9f}; marge exacte : {100 * measured['headroom']:.9f}%")
                print("Historique fourni par l'utilisateur : 1.939667 (comparaison, jamais injecté dans le score).")
                print("Même fichier que le b4 de votre dernier Colab :", entry["tokenizer_sha256"] == HISTORICAL_B4_SHA256)
            else:
                entry["status"] = "measured"
                print("Décision :", "; ".join(selection_reasons(entry, history[0])) or "amélioration admissible")
            history.append(entry)
        except Exception as exc:
            history.append({"config": config, "status": "failed", "error": f"{type(exc).__name__}: {exc}"})
            save_progress(history, context, session_dir)
            if len(history) == 1:
                raise RuntimeError("Baseline failed. Fix the error, then rerun; no alternative trained.") from exc
            print("ÉCHEC enregistré, poursuite du prochain essai :", exc)
        save_progress(history, context, session_dir)
    return history, choose_best(history)

In [ ]:
# Cellule longue. Relancer après interruption avec le même RUN_LABEL et le même stockage.
history, best = run_search(CONFIGS, CONTEXT, SESSION_DIR)
print("\nRecherche terminée. Rapports sauvegardés dans", SESSION_DIR)

## Comparaison et décision — pour examen uniquement

`delta_vs_b4` négatif signifie un score inférieur au b4 de **cette recherche**.
`recommended` signifie « retenu par notre politique locale », pas « adopté dans la soumission ».
La marge est un pourcentage : 1,79 signifie 1,79 %, pas 179 %.
Les colonnes par langue sont dans le CSV et tous les rapports officiels sont conservés.

In [ ]:
from IPython.display import HTML, display
with (SESSION_DIR / "comparison.csv").open(encoding="utf-8", newline="") as stream:
    comparison = list(csv.DictReader(stream))
columns = ["candidate", "status", "score", "delta_vs_b4", "headroom_percent", "guardrail_penalty", "recommended", "reason"]
table = "<table><thead><tr>" + "".join(f"<th>{html.escape(c)}</th>" for c in columns) + "</tr></thead><tbody>"
for row in comparison:
    table += "<tr>" + "".join(f"<td>{html.escape(row[c])}</td>" for c in columns) + "</tr>"
display(HTML(table + "</tbody></table>"))
print("\nRéférence mesurée :", metric_summary(history[0])["score"])
print("Marge minimale (valeur exacte) :", metric_summary(history[0])["headroom"])
print("Candidat retenu pour examen :", best["config"]["name"])
print("Score complet :", best["official"]["score"])
print("SHA-256 :", best["tokenizer_sha256"])
if best is history[0]:
    print("AUCUNE amélioration admissible : référence b4 conservée pour cette recherche.")
else:
    delta = best["official"]["score"] - history[0]["official"]["score"]
    print(f"Variation du score : {delta:+.9f} ({100 * delta / history[0]['official']['score']:+.3f}%)")
print("Aucun tokenizer soumis, README ou metadata n'a été remplacé.")
print("CSV à partager :", SESSION_DIR / "comparison.csv")

In [ ]:
# Archive de diagnostic légère : rapports uniquement, aucun corpus ni modèle.
DOWNLOAD_REPORTS = False  # True puis réexécuter cette cellule pour télécharger dans Colab.
DOWNLOAD_BEST_TOKENIZER = False  # Optionnel : copie pour examen, PAS une soumission.

archive_path = SESSION_DIR / "review_reports.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ("run_context.json", "search_history.json", "comparison.csv"):
        archive.write(SESSION_DIR / name, name)
    for config in CONFIGS:
        directory = SESSION_DIR / config["name"]
        for name in ("manifest.json", "training.json", "official_report.json", "evaluation.json"):
            path = directory / name
            if path.is_file():
                archive.write(path, f"{config['name']}/{name}")
print("À partager pour l'analyse :", archive_path)
print("Fichier candidat (non soumis) :", Path(best["directory"]) / "tokenizer.json")

if DOWNLOAD_REPORTS or DOWNLOAD_BEST_TOKENIZER:
    try:
        from google.colab import files
    except ImportError:
        print("Hors Colab : récupérer les fichiers aux chemins affichés ci-dessus.")
    else:
        if DOWNLOAD_REPORTS:
            files.download(str(archive_path))
        if DOWNLOAD_BEST_TOKENIZER:
            model_path = Path(best["directory"]) / "tokenizer.json"
            verify_saved_candidate(model_path.parent, {"config": best["config"], "context": CONTEXT})
            if sha256_file(model_path) != best["tokenizer_sha256"]:
                raise ValueError("Le modèle a changé depuis son évaluation")
            files.download(str(model_path))

## Ce qu'il faut partager ensuite

Envoyer **`review_reports.zip`** ou **`comparison.csv`**, et préciser si la recherche a été interrompue.
Ne pas envoyer le cache ou les corpus. Garder les modèles sur Drive pour ne pas perdre les octets mesurés.

Après analyse seulement, nous pourrons décider de conserver b4, d'approfondir une piste ou de proposer
un remplacement. Aucun score de test caché n'est prédit ici.
Ce notebook d'exploration se trouve hors de `submissions/` : ne pas l'ajouter tel quel à une PR de participation.
Si un modèle est adopté plus tard, son entraînement et ses résultats devront être documentés dans le notebook
de soumission, avec exactement le fichier évalué.

### Références
- [Checker officiel fixé](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge/blob/75578f2400c39b1f8e31ce7e7104b37fbc470d11/starter/utils.py)
- [Données officielles publiques](https://huggingface.co/datasets/Similoluwa/african-multilingual-tokenizer-challenge/tree/v1.0.0)
- [Règles de participation](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge/blob/main/CONTRIBUTING.md)
- [Notebook de soumission b4, avant cette recherche](https://github.com/maick-code/airf-multilingual-tokenizer-challenge/blob/c564c56005055c8d2823ce5753b2dc6255117403/submissions/maick-dane-nkou/notebook.ipynb)